## Modelling voor klachten train set
### Notebook Modelling klachten classificatie voor correcte expertise

#### model getraind op Omschrijving en bedrijfsantwoord

In [1]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
import nltk

In [14]:

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from scipy.sparse import hstack
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
from scipy.stats import uniform, randint
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from wordcloud import WordCloud
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.metrics import classification_report

In [3]:
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Gebruiker\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Gebruiker\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Gebruiker\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
RANDOM_STATE = 8758

In [5]:
#preprocessing
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

train = pd.read_csv("klacht_train.csv")
test = pd.read_csv("klacht_test.csv")

train['Omschrijving_Cleaned'] = train['Omschrijving'].apply(preprocess_text)
test['Omschrijving_Cleaned'] = test['Omschrijving'].apply(preprocess_text)

y_train = train['Product']
y_test = test['Product']

In [6]:
#search functie
def run_search(pipeline, parameter_distributie, parameter_grid, name):
    randomized_search = RandomizedSearchCV(
        pipeline,
        param_distributions=parameter_distributie,
        n_iter=20,
        cv=5,
        scoring='precision_macro',
        n_jobs=6,
        random_state=42,
        verbose=2
    )
    randomized_search.fit(train['Omschrijving_Cleaned'], y_train)
    print(f"Randomized {name}:", randomized_search.best_params_)

    grid_search = GridSearchCV(
        pipeline,
        param_grid = parameter_grid(randomized_search.best_params_),
        cv=5,
        scoring='precision_macro',
        n_jobs=6,
        verbose=2
    )
    grid_search.fit(train['Omschrijving_Cleaned'], y_train)
    print(f"Grid {name}:", grid_search.best_params_)

    print(classification_report(
        y_test,
        grid_search.predict(test['Omschrijving_Cleaned'])
    ))

In [7]:

train = pd.read_csv("klacht_train.csv")
test  = pd.read_csv("klacht_test.csv")

X_train_text = train["Omschrijving"].str.lower()
X_test_text  = test["Omschrijving"].str.lower()
X_train_comb = (train["Omschrijving"] + " " + train["Antwoord_bedrijf"]).str.lower()
X_test_comb  = (test["Omschrijving"]  + " " + test["Antwoord_bedrijf"]).str.lower()
y_train = train["Product"]
y_test  = test["Product"]



In [8]:
classes = np.unique(y_train)
cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
cw_dict = {cls: w for cls, w in zip(classes, cw)}

In [9]:
# vectorizers
char_vect = TfidfVectorizer(analyzer="char", ngram_range=(3,5), min_df=5, max_features=100_000)
word_vect = TfidfVectorizer(analyzer="word", ngram_range=(1,2), min_df=3, max_features=80_000,
                            sublinear_tf=True, stop_words="english")

In [10]:
# fitting vectorizer op traindata, transformatie train en test
Xtr_char = char_vect.fit_transform(X_train_text)
Xte_char = char_vect.transform(X_test_text)
Xtr_word = word_vect.fit_transform(X_train_comb)
Xte_word = word_vect.transform(X_test_comb)

Xtr = hstack([Xtr_char, Xtr_word]).tocsr()
Xte = hstack([Xte_char, Xte_word]).tocsr()

In [11]:
# linearSVC
svc = LinearSVC(C=1.0, class_weight=cw_dict)
svc.fit(Xtr, y_train)
svc_pred = svc.predict(Xte)


print("\nLinearSVC")
print(classification_report(y_test, svc_pred, digits=2))


LinearSVC
                    precision    recall  f1-score   support

      Bankrekening       0.86      0.84      0.85       240
Consumentenkrediet       0.85      0.63      0.73       156
        Creditcard       0.81      0.83      0.82       299
         Hypotheek       0.93      0.95      0.94       567
           Incasso       0.85      0.90      0.87       616
Kredietregistratie       0.87      0.86      0.86       504

          accuracy                           0.87      2382
         macro avg       0.86      0.83      0.84      2382
      weighted avg       0.87      0.87      0.87      2382



In [12]:
# logistic regression
logreg = LogisticRegression(C=2.0, class_weight=cw_dict, max_iter=3000, solver="liblinear")
logreg.fit(Xtr, y_train)
log_pred = logreg.predict(Xte)


print("\nLogisticRegression")
print(classification_report(y_test, log_pred, digits=2))

c:\Users\Gebruiker\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(



LogisticRegression
                    precision    recall  f1-score   support

      Bankrekening       0.85      0.87      0.86       240
Consumentenkrediet       0.86      0.68      0.76       156
        Creditcard       0.81      0.81      0.81       299
         Hypotheek       0.93      0.94      0.94       567
           Incasso       0.85      0.89      0.87       616
Kredietregistratie       0.85      0.85      0.85       504

          accuracy                           0.87      2382
         macro avg       0.86      0.84      0.85      2382
      weighted avg       0.87      0.87      0.87      2382



In [13]:
# calibratie van svc voor ensemble met logistic regression
cal_svc = CalibratedClassifierCV(LinearSVC(C=1.0, class_weight=cw_dict),
                                 method="sigmoid", cv=3)

cal_svc.fit(Xtr, y_train)
proba_svc = cal_svc.predict_proba(Xte)
proba_log = None
try:
    proba_log = logreg.predict_proba(Xte)
except Exception:
    pass

if proba_log is not None:

    avg_proba = (proba_svc + proba_log) / 2.0
    ensemble_pred = classes[avg_proba.argmax(axis=1)]
    print("\nCalibrated SVC & LogReg")
    print(classification_report(y_test, ensemble_pred, digits=2))
else:
    print("\nNiet beschikbaar, sla ensemble over.")


Calibrated SVC & LogReg
                    precision    recall  f1-score   support

      Bankrekening       0.86      0.87      0.86       240
Consumentenkrediet       0.88      0.64      0.74       156
        Creditcard       0.83      0.82      0.82       299
         Hypotheek       0.93      0.95      0.94       567
           Incasso       0.85      0.90      0.87       616
Kredietregistratie       0.86      0.85      0.86       504

          accuracy                           0.87      2382
         macro avg       0.87      0.84      0.85      2382
      weighted avg       0.87      0.87      0.87      2382



##### Model getest op test data met en zonder bedrijfsantwoorden om accuratie van voorspelling weer te geven 
##### bij een model dat is getraind op bedrijfsantwoord en omschrijving en word toegepast op alleen omschrijving

In [15]:
# test omschrijving en antwoord
Xte_char = char_vect.transform(X_test_text)
Xte_word = word_vect.transform(X_test_comb)
Xte_full = hstack([Xte_char, Xte_word]).tocsr()
y_pred_full = svc.predict(Xte_full)

In [16]:
# test alleen omschrijving
X_test_comb_empty = (test["Omschrijving"] + " ").str.lower()
Xte_word_empty = word_vect.transform(X_test_comb_empty)
Xte_desc_only = hstack([Xte_char, Xte_word_empty]).tocsr()
y_pred_desc_only = svc.predict(Xte_desc_only)

In [17]:
# vergelijk tests
print("Evaluatie met Omschrijving en Antwoord")
print(classification_report(y_test, y_pred_full, digits=2))

print("Evaluatie met alleen Omschrijving")
print(classification_report(y_test, y_pred_desc_only, digits=2))


Evaluatie met Omschrijving en Antwoord
                    precision    recall  f1-score   support

      Bankrekening       0.86      0.84      0.85       240
Consumentenkrediet       0.85      0.63      0.73       156
        Creditcard       0.81      0.83      0.82       299
         Hypotheek       0.93      0.95      0.94       567
           Incasso       0.85      0.90      0.87       616
Kredietregistratie       0.87      0.86      0.86       504

          accuracy                           0.87      2382
         macro avg       0.86      0.83      0.84      2382
      weighted avg       0.87      0.87      0.87      2382

Evaluatie met alleen Omschrijving
                    precision    recall  f1-score   support

      Bankrekening       0.85      0.85      0.85       240
Consumentenkrediet       0.82      0.64      0.72       156
        Creditcard       0.81      0.82      0.82       299
         Hypotheek       0.93      0.95      0.94       567
           Incasso     